In [0]:
import __init__
from src.config.config_store import *

In [0]:
pip install -U polygon-api-client

In [0]:
helper = UCSetup(spark, dbutils)

In [0]:
def fetch_minute_bars_to_spark(spark, client, ticker, multiplier, start_date, end_date, lmt):
    # 1) Fetch aggregates (minute bars)
    aggs = []
    for a in client.list_aggs(
        ticker,
        multiplier,
        timespan,
        from_=start_date,
        to=end_date,
        limit=lmt
    ):
        aggs.append(a)

    # 2) Convert SDK objects to dicts with canonical Polygon keys
    # Polygon returns fields like o,h,l,c,v,vw,t,n,otc; SDK models may expose open/high/low/close/volume/vwap/timestamp/transactions
    aggs_dicts = []
    for a in aggs:
        d = a.__dict__ if hasattr(a, "__dict__") else (a._asdict() if hasattr(a, "_asdict") else dict(a))
        aggs_dicts.append({
            "open": d.get("open", d.get("o")),
            "high": d.get("high", d.get("h")),
            "low":  d.get("low",  d.get("l")),
            "close": d.get("close", d.get("c")),
            "volume": d.get("volume", d.get("v")),
            "vwap": d.get("vwap", d.get("vw")),
            "timestamp": d.get("timestamp", d.get("t")),
            "transactions": d.get("transactions", d.get("n")),
            # otc in raw JSON is boolean; your schema uses StringType, so cast to 'true'/'false' strings
            "otc": str(d.get("otc")).lower() if d.get("otc") is not None else None,
        })

    # 3) pandas DataFrame
    df_pd = pd.DataFrame(aggs_dicts)

    # 4) Spark schema (matches your request)
    schema = StructType([
        StructField("open", DoubleType(), True),
        StructField("high", DoubleType(), True),
        StructField("low", DoubleType(), True),
        StructField("close", DoubleType(), True),
        StructField("volume", LongType(), True),
        StructField("vwap", DoubleType(), True),
        StructField("timestamp", LongType(), True),
        StructField("transactions", LongType(), True),
        StructField("otc", StringType(), True),
    ])

    # 5) Set Spark session timezone to IST and create Spark DataFrame
    spark.conf.set("spark.sql.session.timeZone", "Asia/Kolkata")
    spark_df = (
        spark.createDataFrame(df_pd, schema=schema)
             .withColumn("TimestampIst", (col("timestamp") / 1000).cast("double"))
             .withColumn("TimestampIst", to_timestamp(from_unixtime(col("TimestampIst"))))
    )

    # Compute formatted date strings
    start_int = int(datetime.strptime(start_date, "%Y-%m-%d").strftime("%Y%m%d"))
    end_int   = int(datetime.strptime(end_date, "%Y-%m-%d").strftime("%Y%m%d"))

    # Build output path
    landing_zone = helper.get_paths()["landing_zone"]
    out_path = f"{landing_zone}/{ticker}_{timespan}_{start_int}_{end_int}"
    print(f"Saving CSV to: {out_path}")

    # Write dataframe to CSV
    spark_df.write \
        .mode("overwrite") \
        .option("header", True) \
        .csv(out_path)


In [0]:
client = RESTClient(api_key=APIKEY)
ticker = "AAPL"
timespan = "minute"
multiplier = 1
start_date = "2025-08-19"
end_date = "2025-08-21"
lmt=50000
fetch_minute_bars_to_spark(spark, client, ticker, multiplier, start_date, end_date, lmt)

In [0]:
df = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("dbfs:/Volumes/dev/infra/pipeline_artifacts/landing_zone/AAPL_minute_20250919_20250920")
)
# Saving CSV to: dbfs:/Volumes/dev/infra/pipeline_artifacts/landing_zone/AAPL_minute_20250819_20250820
df.display()

In [0]:
import json
from datetime import datetime

# --- Get/save JSON response ---
resp = client.get_aggs(
    ticker="AAPL",
    multiplier=1,
    timespan="month",
    from_="2025-09-15",
    to="2025-09-19",
    limit=50000,
    raw=True
)
data = json.loads(resp.data)

# --- Build output path (matching CSV logic) ---
start_int = int(datetime.strptime(start_date, "%Y-%m-%d").strftime("%Y%m%d"))
end_int   = int(datetime.strptime(end_date, "%Y-%m-%d").strftime("%Y%m%d"))

landing_zone = helper.get_paths()["landing_zone"]
json_out_path = f"{landing_zone}/{ticker}_{timespan}_{start_int}_{end_int}.json"

print(f"Saving JSON to: {json_out_path}")

# json.dumps returns string; dbutils.fs.put writes to the volume
dbutils.fs.put(
    json_out_path,
    json.dumps(data, ensure_ascii=False, indent=2),
    overwrite=True
)

In [0]:
import json

json_path = "dbfs:/Volumes/dev/infra/pipeline_artifacts/landing_zone/AAPL_minute_20250819_20250820.json"
json_str = dbutils.fs.head(json_path, 10000000)  # read up to 10MB (adjust if needed)
data = json.loads(json_str)
print(data)

In [0]:
import pyspark.sql.functions as sf

schema = """
    open DOUBLE,
    high DOUBLE,
    low DOUBLE,
    close DOUBLE,
    volume LONG,
    vwap DOUBLE,
    timestamp LONG,
    transactions LONG,
    otc STRING,
    TimestampIst STRING
"""

csv_path = "dbfs:/Volumes/dev/infra/pipeline_artifacts/landing_zone/AAPL_minute_*.csv"

df_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("maxFilesPerTrigger", 1)
        .schema(schema)
        .load(csv_path)
        .withColumn("load_time", sf.current_timestamp())
        # UC: use _metadata.file_path instead of input_file_name()
        .withColumn("source_file", sf.col("_metadata.file_path"))
)

bronze_table = "dev.bronze.aapl_minutes"

bronze_writer = (
    df_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", "dbfs:/Volumes/dev/infra/pipeline_artifacts/checkpoint/aapl_minutes_bronze")
        .queryName("aapl_minutes_bronze")
        .trigger(availableNow=True)
        .toTable(bronze_table)
)

In [0]:
%sql
select *
from dev.bronze.aapl_minutes